In [68]:
import pandas as pd
from functions.eval import *
from tqdm.notebook import tqdm
tqdm.pandas()

In [69]:
model_pred_col = "Camelbert-MSA"
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa-sentiment"
# model_name = "PRAli22/AraBert-Arabic-Sentiment-Analysis"
# model_pred_col = "AraBert"

In [70]:
eval_df = pd.read_csv("data/hard_rationale/hard_rationale_ensemble_" + model_pred_col + ".csv")

In [71]:
hard_rationale_choices = {
    "elbow": {
        "elbow_method": ["simple-lmethod", "kneedle", "dfdt", "lmethod"],
    },
    "top_n": {
        "n": [3, 5, 10, 20],
    },
    "threshold": {
        "k": [0.1, 0.3, 0.5, 0.7],
    }
}

In [27]:
for rationale_type in hard_rationale_choices.keys():
    for choice, values in hard_rationale_choices[rationale_type].items():
        for value in values:
            eval_df["nb_selected_tokens_" + rationale_type + "_" + str(value)] = \
                eval_df.progress_apply(lambda row: len(hard_rationale_selection(eval(row["EnsembleXAI_LIME_SHAP_mean"]), 
                                                    method=rationale_type, **{choice: value})), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

...

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [28]:
for rationale_type in hard_rationale_choices.keys():
    for choice, values in hard_rationale_choices[rationale_type].items():
        for value in values:
            eval_df["%_tokens_input_" + rationale_type + "_" + str(value)] = \
                eval_df.progress_apply(lambda row: int(row["nb_selected_tokens_" + rationale_type + "_" + str(value)]) /
                                        len(eval(row["EnsembleXAI_LIME_SHAP_mean"])), axis=1)

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

...

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

  0%|          | 0/3033 [00:00<?, ?it/s]

In [29]:
eval_df.to_csv("data/hard_rationale/hard_rationale_ensemble_" + model_pred_col + ".csv", index=False)

Rationale selection runtime benchmarking

In [73]:
score_data = [
    eval(x)
    for x in eval_df["EnsembleXAI_LIME_SHAP_mean"]
]

In [74]:
import time
import random
import gc

configs = [
    ("simple-lmethod", "elbow", {"elbow_method": "simple-lmethod"}),
    ("kneedle", "elbow", {"elbow_method": "kneedle"}),
    ("dfdt", "elbow", {"elbow_method": "dfdt"}),
    ("lmethod", "elbow", {"elbow_method": "lmethod"}),
    ("top-3", "top_n", {"n": 3}),
    ("top-5", "top_n", {"n": 5}),
    ("top-10", "top_n", {"n": 10}),
    ("top-20", "top_n", {"n": 20}),
    ("threshold-0.1", "threshold", {"k": 0.1}),
    ("threshold-0.3", "threshold", {"k": 0.3}),
    ("threshold-0.5", "threshold", {"k": 0.5}),
    ("threshold-0.7", "threshold", {"k": 0.7}),
]

REPEATS = 30

records = []

for repeat in range(REPEATS):

    # Randomize execution order to avoid systematic order effects
    shuffled = configs.copy()
    random.shuffle(shuffled)

    for name, method, kwargs in shuffled:

        start = time.perf_counter_ns()

        for token_weights in score_data:
            hard_rationale_selection(
                token_weights,
                method=method,
                **kwargs
            )

        elapsed_ns = time.perf_counter_ns() - start

        records.append({
            "method": name,
            "repeat": repeat,
            "total_ms": elapsed_ns / 1e6,
            "per_instance_us":
                elapsed_ns / len(score_data) / 1e3
        })

runtime = pd.DataFrame(records)

In [ ]:
# sort values using configs order
runtime = runtime.set_index("method").loc[[name for name, _, _ in configs]].reset_index()

In [76]:
summary = (
    runtime.groupby("method")["per_instance_us"]
    .agg(
        mean="mean",
        std="std"
    )
)
summary

,mean,std
method,,
dfdt,218.633597,179.271518
kneedle,110.493307,85.078639
lmethod,766.382768,549.550140
simple-lmethod,22.622428,19.203190
threshold-0.1,7.052946,4.229904
threshold-0.3,6.920644,3.965779
threshold-0.5,7.031451,4.044526
threshold-0.7,6.840299,4.134164
top-10,6.439009,3.920093


In [77]:
summary.to_csv(
    "data/hard_rationale/hard_rationale_runtime_" + model_pred_col + ".csv",
    # index=False
)